# Retrieval Baseline Results — Medium Model

Compares BM25, Dense, and Cross-Encoder pipelines on the `medium` Whisper model.
All three evaluated at top-10 on the same query set.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

PROJECT_ROOT = Path("../")
METRICS_DIR  = PROJECT_ROOT / "data" / "metrics"
EVAL_DIR     = PROJECT_ROOT / "data" / "eval"
CP_DIR       = EVAL_DIR / "checkpoints"

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

## 1. Load latest summary for each pipeline

In [ ]:
def latest_csv(directory: Path, prefix: str) -> Path:
    """Return the most recently created CSV matching prefix."""
    candidates = sorted(directory.glob(f"{prefix}*.csv"))
    if not candidates:
        raise FileNotFoundError(f"No {prefix}*.csv in {directory}")
    return candidates[-1]

pipelines = {
    "BM25":          "bm25",
    "Dense":         "dense",
    "Cross-Encoder": "crossencoder",
}

summary_rows = []
per_query = {}

for label, key in pipelines.items():
    s_path = latest_csv(METRICS_DIR / key, "summary_medium_")
    q_path = latest_csv(METRICS_DIR / key, "per_query_medium_")
    s = pd.read_csv(s_path)
    q = pd.read_csv(q_path)
    # Overwrite the pipeline column with the display label (avoid duplicate insert)
    s["pipeline"] = label
    q["pipeline"] = label
    summary_rows.append(s)
    per_query[label] = q

summary = pd.concat(summary_rows, ignore_index=True)
print(summary[["pipeline", "mrr", "recall", "precision", "ndcg"]].to_string(index=False))

## 2. Query set breakdown

In [ ]:
queries_df = pd.read_csv(EVAL_DIR / "retrieval_eval_queries.csv")

flag_counts = queries_df["difficulty_flag"].value_counts()
print("Query difficulty breakdown:")
print(flag_counts.to_string())
print(f"\nTotal queries:         {len(queries_df)}")
print(f"Evaluated (has labels): {len(per_query['Dense'])}")
print(f"  of which difficulty=ok:            {(queries_df['difficulty_flag']=='ok').sum()}")
print(f"  of which difficulty=trivial_overlap: {(queries_df['difficulty_flag']=='trivial_overlap').sum()}")

fig, ax = plt.subplots(figsize=(5, 3.5))
colors = ["#4CAF50", "#FF9800", "#F44336"]
bars = ax.barh(flag_counts.index, flag_counts.values, color=colors[:len(flag_counts)])
ax.bar_label(bars, padding=3)
ax.set_xlabel("Number of queries")
ax.set_title("Query set by difficulty flag")
plt.tight_layout()
plt.show()

## 3. Aggregate metric comparison (all evaluated queries)

In [ ]:
metrics = ["ndcg", "mrr", "recall", "precision"]
metric_labels = ["NDCG@10", "MRR", "Recall@10", "Precision@10"]
pipe_labels = list(pipelines.keys())
colors_pipe = ["#607D8B", "#1976D2", "#E53935"]

x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(9, 4.5))
for i, (label, color) in enumerate(zip(pipe_labels, colors_pipe)):
    vals = [float(summary.loc[summary["pipeline"] == label, m].iloc[0]) for m in metrics]
    bars = ax.bar(x + i * width, vals, width, label=label, color=color, alpha=0.88)
    ax.bar_label(bars, fmt="%.3f", padding=2, fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(metric_labels)
ax.set_ylabel("Score")
ax.set_ylim(0, 0.70)
ax.set_title("Retrieval baseline comparison — medium model, top-10")
ax.legend()
plt.tight_layout()
plt.show()

## 4. NDCG@10 on ok-only queries vs. full set

In [ ]:
# Map query_id → difficulty_flag
flag_map = dict(zip(queries_df["query_id"], queries_df["difficulty_flag"]))

rows = []
for label, q in per_query.items():
    q2 = q.copy()
    q2["flag"] = q2["query_id"].map(flag_map).fillna("unknown")
    all_ndcg    = q2["ndcg"].mean()
    ok_ndcg     = q2.loc[q2["flag"] == "ok", "ndcg"].mean()
    triv_ndcg   = q2.loc[q2["flag"] == "trivial_overlap", "ndcg"].mean()
    rows.append({"pipeline": label, "All queries": all_ndcg,
                 "ok only": ok_ndcg, "trivial_overlap only": triv_ndcg})

subset_df = pd.DataFrame(rows).set_index("pipeline")
print(subset_df.round(4).to_string())

fig, ax = plt.subplots(figsize=(7, 4))
subset_df[["All queries", "ok only", "trivial_overlap only"]].plot(
    kind="bar", ax=ax, color=["#90A4AE", "#1976D2", "#FF9800"], alpha=0.88
)
ax.set_ylabel("NDCG@10")
ax.set_title("NDCG@10 by query difficulty subset")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.set_ylim(0, 0.85)
ax.legend(loc="upper left")
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", padding=2, fontsize=8)
plt.tight_layout()
plt.show()

## 5. Per-query NDCG distribution (box plots)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
data = [per_query[label]["ndcg"].values for label in pipe_labels]
bp = ax.boxplot(data, labels=pipe_labels, patch_artist=True, notch=False,
                medianprops=dict(color="black", linewidth=2))
for patch, color in zip(bp["boxes"], colors_pipe):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
ax.set_ylabel("NDCG@10 per query")
ax.set_title("Per-query NDCG@10 distribution")
ax.yaxis.set_minor_locator(mticker.MultipleLocator(0.05))
ax.grid(axis="y", which="both", alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Gain from reranking: Dense → Cross-Encoder (per query)

In [ ]:
dense_q = per_query["Dense"].set_index("query_id")["ndcg"]
ce_q    = per_query["Cross-Encoder"].set_index("query_id")["ndcg"]
common  = dense_q.index.intersection(ce_q.index)

gain = (ce_q[common] - dense_q[common]).sort_values()

improved  = (gain > 0).sum()
degraded  = (gain < 0).sum()
unchanged = (gain == 0).sum()
print(f"CE improved over Dense:  {improved} queries")
print(f"CE degraded vs Dense:    {degraded} queries")
print(f"Unchanged:               {unchanged} queries")
print(f"Mean gain:               {gain.mean():+.4f}")

fig, ax = plt.subplots(figsize=(9, 3.5))
colors_gain = ["#E53935" if v < 0 else "#1976D2" for v in gain.values]
ax.bar(range(len(gain)), gain.values, color=colors_gain, alpha=0.8, width=1.0)
ax.axhline(0, color="black", linewidth=0.8)
ax.axhline(gain.mean(), color="orange", linewidth=1.2, linestyle="--", label=f"mean gain {gain.mean():+.3f}")
ax.set_xlabel("Queries (sorted by gain)")
ax.set_ylabel("ΔNDCG@10  (CE − Dense)")
ax.set_title("Per-query NDCG gain from cross-encoder reranking")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Hit rate at various K

In [ ]:
# Reload per-query results (with ranked lists) to compute recall@K curves
results_dir = PROJECT_ROOT / "data" / "retrieval_results"

def recall_at_k(results_df: pd.DataFrame, k: int) -> float:
    """Mean recall@k across all queries in results_df."""
    per_q = []
    for qid, grp in results_df.groupby("query_id"):
        top_k = grp.sort_values("rank").head(k)
        total_rel = (grp["relevance"] > 0).sum()
        if total_rel == 0:
            continue
        hits = (top_k["relevance"] > 0).sum()
        per_q.append(hits / total_rel)
    return float(np.mean(per_q)) if per_q else 0.0

result_files = {
    "BM25":          latest_csv(results_dir / "bm25",          "results_medium_"),
    "Dense":         latest_csv(results_dir / "dense",         "results_medium_"),
    "Cross-Encoder": latest_csv(results_dir / "crossencoder",  "results_medium_"),
}

results = {label: pd.read_csv(path) for label, path in result_files.items()}

ks = [1, 3, 5, 10]
fig, ax = plt.subplots(figsize=(7, 4))
for (label, res_df), color in zip(results.items(), colors_pipe):
    recalls = [recall_at_k(res_df, k) for k in ks]
    ax.plot(ks, recalls, marker="o", label=label, color=color, linewidth=2)

ax.set_xlabel("K")
ax.set_ylabel("Recall@K")
ax.set_title("Recall@K curve — medium model")
ax.set_xticks(ks)
ax.set_ylim(0, 0.85)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Summary table (thesis-ready)

In [ ]:
table = summary[["pipeline", "ndcg", "mrr", "recall", "precision"]].copy()
table.columns = ["Pipeline", "NDCG@10", "MRR", "Recall@10", "Precision@10"]
table = table.set_index("Pipeline")
table = table.round(4)

# Add relative gain over BM25
bm25_ndcg = table.loc["BM25", "NDCG@10"]
table["ΔNDCG vs BM25"] = (table["NDCG@10"] - bm25_ndcg).round(4)

print("=" * 60)
print("RETRIEVAL BASELINE RESULTS — medium model, top-10")
print("=" * 60)
print(table.to_string())
print("\nNote: evaluated on 96 queries with at least one positive label.")
print(f"      73 ok | 23 trivial_overlap | 99 no_positives (excluded)")